# 02 · Data understanding (EDA)

**Input:** `outputs/data/reviews_clean.pkl`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
from collections import Counter

import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.config import LABELS, PALETTE, SEED, setup_notebook
from src.data import load_clean
from src.text import NEGATIONS, clean_text, tokenize

setup_notebook()
nltk.download('stopwords', quiet=True)
df = load_clean()
print(f'{len(df):,} reviews after cleaning')

## 1. Class distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

score_counts = df['Score'].value_counts().sort_index()
axes[0].bar(score_counts.index.astype(str), score_counts.values, color=sns.color_palette('RdYlGn', 5))
for i, v in enumerate(score_counts.values):
    axes[0].text(i, v, f'{v:,}\n{v / len(df):.1%}', ha='center', va='bottom', fontsize=9)
axes[0].set(title='Star rating distribution (Score)', xlabel='Stars', ylabel='Reviews', ylim=(0, score_counts.max() * 1.18))

label_counts = df['sentiment'].value_counts().reindex(LABELS)
axes[1].bar(LABELS, label_counts.values, color=[PALETTE[l] for l in LABELS])
for i, v in enumerate(label_counts.values):
    axes[1].text(i, v, f'{v:,}\n{v / len(df):.1%}', ha='center', va='bottom', fontsize=9)
axes[1].set(title='Sentiment label distribution', xlabel='Label', ylabel='Reviews', ylim=(0, label_counts.max() * 1.18))
plt.tight_layout()
plt.show()

print(f'Imbalance ratio positive : neutral = {label_counts["positive"] / label_counts["neutral"]:.1f} : 1')

## 2. Text length

In [ ]:
df['n_words_text'] = df['Text'].str.split().str.len()
df['n_words_summary'] = df['Summary'].str.split().str.len()
display(df[['n_words_text', 'n_words_summary']]
        .describe(percentiles=[.5, .75, .9, .95, .99]).T.round(1))

df['n_words_clip'] = df['n_words_text'].clip(upper=500)
fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
sns.histplot(data=df, x='n_words_clip', hue='sentiment', hue_order=LABELS, palette=PALETTE, bins=60,
             stat='density', common_norm=False, element='step', fill=False, ax=axes[0])
axes[0].set(title='Word count by label', xlabel='Words in Text (clipped at 500)')

sns.boxplot(data=df, x='Score', y='n_words_text', hue='Score', palette='RdYlGn', legend=False, showfliers=False, ax=axes[1])
axes[1].set(title='Word count by stars (outliers hidden)', xlabel='Stars', ylabel='Words')

lens = np.sort(df['n_words_text'].values)
grid = np.arange(0, 501)
axes[2].plot(grid, np.searchsorted(lens, grid, side='right') / len(lens), color='#4c72b0')
for L in [128, 200, 256]:
    cov = (lens <= L).mean()
    axes[2].axvline(L, ls='--', color='gray', lw=1)
    axes[2].text(L + 4, cov - 0.08, f'{L} words: {cov:.1%}', fontsize=9)
axes[2].set(title='Share of reviews with at most L words', xlabel='L (words)', ylabel='Cumulative share', ylim=(0, 1.02))
plt.tight_layout()
plt.show()

## 3. Change over time

In [ ]:
yearly = df.groupby(df['date'].dt.year).agg(n=('Score', 'size'), mean_score=('Score', 'mean'),
                                            pct_neg=('label', lambda s: (s == 0).mean()))
fig, ax1 = plt.subplots(figsize=(13, 4.5))
ax1.bar(yearly.index, yearly['n'], color='#4c72b0', alpha=.75, label='Reviews')
ax1.set(xlabel='Year', ylabel='Reviews', title='Reviews and mean score per year')
ax2 = ax1.twinx()
ax2.plot(yearly.index, yearly['mean_score'], 'o-', color='#dd8452', label='Mean score')
ax2.set_ylabel('Mean score')
ax2.grid(False)
fig.legend(loc='upper left', bbox_to_anchor=(0.07, 0.88))
plt.tight_layout()
plt.show()
yearly.T.round(3)

## 4. Helpfulness by star rating

In [ ]:
helpful = df[df['HelpfulnessDenominator'] >= 5].copy()
helpful['helpful_ratio'] = helpful['HelpfulnessNumerator'] / helpful['HelpfulnessDenominator']

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(data=helpful, x='Score', y='helpful_ratio', hue='Score', palette='RdYlGn', legend=False, errorbar='se', ax=axes[0])
axes[0].set(title=f'Mean helpful ratio (reviews with at least 5 votes, n={len(helpful):,})',
            xlabel='Stars', ylabel='Numerator / Denominator')
share = df.assign(has_vote=df['HelpfulnessDenominator'] > 0).groupby('Score')['has_vote'].mean()
axes[1].bar(share.index.astype(str), share.values, color=sns.color_palette('RdYlGn', 5))
axes[1].set(title='Share of reviews with at least 1 vote', xlabel='Stars', ylabel='Share')
plt.tight_layout()
plt.show()

## 5. Users and products

In [ ]:
per_user = df['UserId'].value_counts()
per_product = df['ProductId'].value_counts()
print(f'Users: {per_user.size:,} | Products: {per_product.size:,}')
print(f'Users with a single review: {(per_user == 1).mean():.1%} | Products with at most 5 reviews: {(per_product <= 5).mean():.1%}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(per_user.values, bins=np.logspace(0, np.log10(per_user.max() + 1), 40), color='#4c72b0')
axes[0].set(xscale='log', yscale='log', title='Reviews per user', xlabel='Reviews', ylabel='Users')
axes[1].hist(per_product.values, bins=np.logspace(0, np.log10(per_product.max() + 1), 40), color='#55a868')
axes[1].set(xscale='log', yscale='log', title='Reviews per product', xlabel='Reviews', ylabel='Products')
plt.tight_layout()
plt.show()

## 6. Noise in the text

In [ ]:
noise = pd.Series({
    'Contains HTML tags (<br />, <a href...>)': df['Text'].str.contains(r'<[^>]+>', regex=True).mean(),
    'Contains URLs (http, www)': df['Text'].str.contains(r'http|www\.', regex=True).mean(),
    'Contains encoded HTML characters (&amp; &quot; ...)': df['Text'].str.contains(r'&[a-z]+;', regex=True).mean(),
}, name='Share of reviews').map('{:.2%}'.format)
display(noise.to_frame())

sample_html = df.loc[df['Text'].str.contains('<br', regex=False), 'Text'].iloc[0]
print('BEFORE:', sample_html[:400], '\n')
print('AFTER :', clean_text(sample_html)[:400])

## 7. Vocabulary of each class

In [ ]:
STOPWORDS = set(nltk.corpus.stopwords.words('english')) - NEGATIONS

n_per_class = min(20_000, df['label'].value_counts().min())
eda_sample = pd.concat([g.sample(n_per_class, random_state=SEED) for _, g in df.groupby('label')])
eda_tokens = [[w for w in tokenize(clean_text(t)) if w not in STOPWORDS and w.isalnum()]
              for t in eda_sample['Summary'] + ' ' + eda_sample['Text']]

def top_ngrams(token_lists, n, k=15):
    counter = Counter()
    for toks in token_lists:
        counter.update(' '.join(toks[i:i + n]) for i in range(len(toks) - n + 1))
    return pd.Series(dict(counter.most_common(k)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for row, ngram in enumerate([1, 2]):
    for col, lab in enumerate(LABELS):
        toks = [t for t, s in zip(eda_tokens, eda_sample['sentiment']) if s == lab]
        s = top_ngrams(toks, ngram)
        axes[row, col].barh(s.index[::-1], s.values[::-1], color=PALETTE[lab])
        axes[row, col].set_title(f'Top {"unigram" if ngram == 1 else "bigram"} – {lab}')
plt.suptitle(f'Most frequent words and phrases per class (balanced sample of {n_per_class:,} reviews per class, stop words removed except negations)', y=1.0)
plt.tight_layout()
plt.show()